# Momentum and Adaptive Methods

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 04.03 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/04_momentum_and_adaptive_methods.ipynb)

---

## 🎯 Learning Objective

Derive momentum, Nesterov acceleration, and adaptive per-parameter methods (AdaGrad, RMSProp, Adam, AdamW) and see why Adam dominates in practice.

---

## 📐 Theory

`04.02` and `04.03` showed that ill-conditioning (large $\kappa=L/\mu$) makes plain gradient
descent crawl: the same $\eta$ stable along a steep direction is too small along a shallow one.
Momentum and the adaptive methods below are two independent fixes, and Adam combines both.

### Momentum

**Momentum** accumulates a running average of past gradients into a velocity $\mathbf{v}$, and
steps along it instead of the raw gradient:

$$\mathbf{v}_t = \mu\, \mathbf{v}_{t-1} + \nabla L(\mathbf{w}_{t-1}), \qquad
\mathbf{w}_t = \mathbf{w}_{t-1} - \eta\, \mathbf{v}_t$$

with $\mu\in[0,1)$ (typically $0.9$). Where consecutive gradients agree (the shallow direction
of an elongated bowl), velocity builds up and accelerates progress; where gradients keep
flipping sign (the steep direction), terms cancel and oscillation is damped. Momentum only
*sometimes* fixes ill conditioning — it can still overshoot on the steep direction, visible in
the trajectory below.

### Nesterov, AdaGrad, RMSProp

**Nesterov momentum** evaluates the gradient at a *look-ahead* point
$\mathbf{w}_{t-1}-\eta\mu\,\mathbf{v}_{t-1}$ rather than $\mathbf{w}_{t-1}$ itself — "look
before you leap," giving a provably better convergence rate for convex problems (Nesterov,
1983).

Momentum uses one global step size for every coordinate. **AdaGrad** (Duchi et al., 2011)
instead divides each coordinate's step by its *accumulated* squared-gradient magnitude,
$G_t=G_{t-1}+\nabla L^2$, so frequently-large-gradient coordinates get smaller effective steps
and rare ones get larger — exactly what an ill-conditioned bowl needs. Flaw: $G_t$ only grows,
so the rate decays to zero. **RMSProp** (Hinton, 2012) fixes this with an exponential moving
average instead of a running sum, so old gradients decay away.

### Adam: momentum and per-parameter scaling, together

**Adam** (Kingma & Ba, 2015) keeps exponential moving averages of both the gradient (like
momentum) and its square (like RMSProp):

$$\mathbf{m}_t = \beta_1 \mathbf{m}_{t-1} + (1-\beta_1)\,\mathbf{g}_t, \qquad
\mathbf{v}_t = \beta_2 \mathbf{v}_{t-1} + (1-\beta_2)\,\mathbf{g}_t^2$$

Both start at $\mathbf{0}$, biasing early estimates toward zero: for constant
$\mathbf{g}_t=\mathbf{g}$, $\mathbf{m}_t=(1-\beta_1^t)\mathbf{g}$ — at $t=1,\beta_1=0.9$ that's
a $90\%$ underestimate. Adam divides out the known factor:

$$\hat{\mathbf{m}}_t = \frac{\mathbf{m}_t}{1-\beta_1^t}, \qquad
\hat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1-\beta_2^t}, \qquad
\mathbf{w}_t = \mathbf{w}_{t-1} - \eta\,\frac{\hat{\mathbf{m}}_t}{\sqrt{\hat{\mathbf{v}}_t}+\epsilon}$$

We verify below this correction is exact for constant gradients, and skipping it makes the
*effective* step drift substantially over the first several steps.

### AdamW: weight decay done correctly

**Weight decay** shrinks weights each step and is mathematically identical to L2 regularization
*only* for plain SGD. Adam's adaptive $1/\sqrt{\hat v_t}$ scaling multiplies the L2 gradient
term too when folded into $\mathbf{g}_t$, decaying small-$\hat v_t$ coordinates
disproportionately — an unintended coupling. **AdamW** (Loshchilov & Hutter, 2019) decouples
decay from the adaptive step: apply $\mathbf{w}\leftarrow(1-\eta\lambda)\mathbf{w}$ directly.
Adam-with-L2 and AdamW give measurably different trajectories on an identical problem, verified
below — a real, not cosmetic, distinction.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import torch
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Racing plain SGD, momentum, and Adam from the same start on the same ill-conditioned bowl shows
each method's signature failure mode and fix: SGD crawls, momentum overshoots and oscillates
before settling, and Adam's per-coordinate scaling cuts almost straight to the minimum.

In [ ]:
# Same kind of ill-conditioned bowl as 04.02/04.03 (kappa=25), now racing three optimizers
# from the same start with hand-tuned step sizes for each.
A = np.diag([1.0, 25.0])
grad_fn = lambda w: A @ w
loss_fn = lambda w: 0.5 * w @ A @ w
w0 = np.array([1.0, 1.0])
steps = 40

def run_sgd(lr):
    w = w0.copy(); traj = [w.copy()]
    for _ in range(steps):
        w = w - lr * grad_fn(w)
        traj.append(w.copy())
    return np.array(traj)

def run_momentum(lr, mu):
    w = w0.copy(); v = np.zeros_like(w); traj = [w.copy()]
    for _ in range(steps):
        v = mu * v + grad_fn(w)
        w = w - lr * v
        traj.append(w.copy())
    return np.array(traj)

def run_adam(lr, b1=0.9, b2=0.999, eps=1e-8):
    w = w0.copy(); m = np.zeros_like(w); v = np.zeros_like(w); traj = [w.copy()]
    for t in range(1, steps + 1):
        g = grad_fn(w)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g**2
        m_hat, v_hat = m / (1 - b1**t), v / (1 - b2**t)
        w = w - lr * m_hat / (np.sqrt(v_hat) + eps)
        traj.append(w.copy())
    return np.array(traj)

traj_sgd = run_sgd(0.06)
traj_mom = run_momentum(0.02, 0.8)
traj_adam = run_adam(0.5)

w1g, w2g = np.linspace(-1.2, 1.2, 150), np.linspace(-1.2, 1.2, 150)
W1, W2 = np.meshgrid(w1g, w2g)
Z = 0.5 * (A[0, 0] * W1**2 + A[1, 1] * W2**2)

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.contour(W1, W2, Z, levels=30, cmap="viridis", alpha=0.6)
ax.plot(*traj_sgd.T, "-o", ms=2, color="#C44E52", lw=1, label="SGD (lr=0.06)")
ax.plot(*traj_mom.T, "-o", ms=2, color="#DD8452", lw=1, label="momentum (lr=0.02, mu=0.8)")
ax.plot(*traj_adam.T, "-o", ms=2, color="#4C72B0", lw=1, label="Adam (lr=0.5)")
ax.plot(0, 0, "k*", markersize=16, label="minimum")
ax.set_xlabel("w1 (shallow direction)"); ax.set_ylabel("w2 (steep direction, curvature 25x)")
ax.set_title(f"Same start, {steps} steps each -- three ways of using the same gradients")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

for name, traj in [("SGD", traj_sgd), ("momentum", traj_mom), ("Adam", traj_adam)]:
    print(f"{name:10s} final loss = {loss_fn(traj[-1]):.2e}")
print("\nMomentum overshoots along w2 before damping out (visible oscillation); Adam's per-")
print("coordinate scaling avoids that overshoot entirely by shrinking the effective step where")
print("curvature (and hence squared-gradient magnitude) is largest.")

## 🐍 Implementation from Scratch

We implement momentum, Nesterov, AdaGrad, RMSProp, and Adam as small stateful classes operating
on plain NumPy arrays, isolate the effect of Adam's bias correction on a constant-gradient toy
problem, and cross-verify every optimizer against its exact `torch.optim` counterpart on the
same quadratic.

In [ ]:
class Momentum:
    def __init__(self, lr, mu=0.9):
        self.lr, self.mu, self.v = lr, mu, None
    def step(self, w, g):
        self.v = g.copy() if self.v is None else self.mu * self.v + g
        return w - self.lr * self.v

class Nesterov:  # matches torch.optim.SGD(nesterov=True): update = g + mu*v_new
    def __init__(self, lr, mu=0.9):
        self.lr, self.mu, self.v = lr, mu, None
    def step(self, w, g):
        self.v = g.copy() if self.v is None else self.mu * self.v + g
        return w - self.lr * (g + self.mu * self.v)

class AdaGrad:
    def __init__(self, lr, eps=1e-10):
        self.lr, self.eps, self.G = lr, eps, None
    def step(self, w, g):
        self.G = g**2 if self.G is None else self.G + g**2
        return w - self.lr * g / (np.sqrt(self.G) + self.eps)

class RMSProp:
    def __init__(self, lr, beta=0.99, eps=1e-8):
        self.lr, self.beta, self.eps, self.v = lr, beta, eps, None
    def step(self, w, g):
        self.v = (1 - self.beta) * g**2 if self.v is None else self.beta * self.v + (1 - self.beta) * g**2
        return w - self.lr * g / (np.sqrt(self.v) + self.eps)

class Adam:
    def __init__(self, lr, beta1=0.9, beta2=0.999, eps=1e-8, bias_correct=True):
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.bias_correct, self.m, self.v, self.t = bias_correct, None, None, 0
    def step(self, w, g):
        if self.m is None:
            self.m, self.v = np.zeros_like(w), np.zeros_like(w)
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * g
        self.v = self.b2 * self.v + (1 - self.b2) * g**2
        if self.bias_correct:
            m_hat = self.m / (1 - self.b1**self.t)
            v_hat = self.v / (1 - self.b2**self.t)
        else:
            m_hat, v_hat = self.m, self.v   # the ablation: use raw (biased) moments directly
        return w - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

# Bias correction: what breaks without it? Constant gradient isolates the effect.
g_const = np.array([2.5])
opt_raw, opt_corr = Adam(lr=0.1, bias_correct=False), Adam(lr=0.1, bias_correct=True)
w_raw, w_corr = np.array([0.0]), np.array([0.0])
print(f"{'t':>2} {'step, NO correction':>20} {'step, WITH correction':>22}")
for t in range(1, 6):
    prev_raw, prev_corr = w_raw.copy(), w_corr.copy()
    w_raw = opt_raw.step(w_raw, g_const)
    w_corr = opt_corr.step(w_corr, g_const)
    print(f"{t:2d} {(prev_raw-w_raw)[0]:20.5f} {(prev_corr-w_corr)[0]:22.5f}")
print("Corrected step is EXACTLY lr=0.1 at every t. Uncorrected is 3-6x TOO LARGE instead,")
print("and gets WORSE with t: beta2=0.999 is much closer to 1 than beta1=0.9, so v's bias\n"
      "factor (1-beta2^t) shrinks far slower than m's (1-beta1^t). v sits under a square\n"
      "root in the DENOMINATOR, so an under-corrected (too-small) v makes the step too LARGE.")

# Cross-check every optimizer against torch.optim on the SAME ill-conditioned quadratic
def torch_run(opt_ctor, steps=40):
    w_t = torch.tensor(w0.copy(), dtype=torch.float64, requires_grad=True)
    opt = opt_ctor([w_t])
    A_t = torch.tensor(A)
    for _ in range(steps):
        opt.zero_grad()
        (0.5 * w_t @ A_t @ w_t).backward()
        opt.step()
    return w_t.detach().numpy()

def ours_run(opt, steps=40):
    w = w0.copy()
    for _ in range(steps):
        w = opt.step(w, grad_fn(w))
    return w

checks = [
    ("Momentum", Momentum(lr=0.02, mu=0.8), lambda ps: torch.optim.SGD(ps, lr=0.02, momentum=0.8)),
    ("Nesterov", Nesterov(lr=0.02, mu=0.8), lambda ps: torch.optim.SGD(ps, lr=0.02, momentum=0.8, nesterov=True)),
    ("AdaGrad", AdaGrad(lr=0.5), lambda ps: torch.optim.Adagrad(ps, lr=0.5, eps=1e-10)),
    ("RMSProp", RMSProp(lr=0.05), lambda ps: torch.optim.RMSprop(ps, lr=0.05, alpha=0.99, eps=1e-8)),
    ("Adam", Adam(lr=0.5), lambda ps: torch.optim.Adam(ps, lr=0.5)),
]
print(f"\n{'optimizer':10s} {'match torch.optim':>18}")
for name, ours_opt, torch_ctor in checks:
    w_ours, w_torch = ours_run(ours_opt), torch_run(torch_ctor)
    print(f"{name:10s} {np.allclose(w_ours, w_torch, atol=1e-6)!s:>18}")

## 🤖 Why This Matters for AI

AdamW is the default optimizer for training transformers — GPT, BERT, LLaMA, and essentially
every modern LLM use it, since Adam's per-parameter adaptivity handles the wildly different
gradient scales across embedding, attention, and feed-forward layers far better than plain SGD.
But `torch.optim.Adam(weight_decay=...)` and `torch.optim.AdamW(weight_decay=...)` are
genuinely different algorithms, not aliases — the decoupling argument has a measurable effect
on where training ends up, verified directly below. We then extend the same optimizer class to
a small `torch.nn` regression network to confirm the from-scratch Adam trains a real model.

In [ ]:
# 1) AdamW vs Adam+L2: same weight decay coefficient, genuinely different trajectories
class AdamL2:  # L2: fold weight_decay*w into the gradient BEFORE Adam's moments
    def __init__(self, lr, weight_decay, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.wd, self.b1, self.b2, self.eps = lr, weight_decay, beta1, beta2, eps
        self.m = self.v = None; self.t = 0
    def step(self, w, g):
        g = g + self.wd * w
        if self.m is None:
            self.m, self.v = np.zeros_like(w), np.zeros_like(w)
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * g
        self.v = self.b2 * self.v + (1 - self.b2) * g**2
        m_hat, v_hat = self.m / (1 - self.b1**self.t), self.v / (1 - self.b2**self.t)
        return w - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

class AdamWScratch:  # decoupled decay: shrink w directly, independent of gradient statistics
    def __init__(self, lr, weight_decay, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.wd, self.b1, self.b2, self.eps = lr, weight_decay, beta1, beta2, eps
        self.m = self.v = None; self.t = 0
    def step(self, w, g):
        if self.m is None:
            self.m, self.v = np.zeros_like(w), np.zeros_like(w)
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * g
        self.v = self.b2 * self.v + (1 - self.b2) * g**2
        m_hat, v_hat = self.m / (1 - self.b1**self.t), self.v / (1 - self.b2**self.t)
        w = w - self.lr * self.wd * w
        return w - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

wd = 0.1
opt_l2, opt_adamw = AdamL2(lr=0.1, weight_decay=wd), AdamWScratch(lr=0.1, weight_decay=wd)
w_l2, w_adamw = w0.copy(), w0.copy()
for _ in range(80):
    w_l2 = opt_l2.step(w_l2, grad_fn(w_l2))
    w_adamw = opt_adamw.step(w_adamw, grad_fn(w_adamw))
print(f"Adam+L2 final w={np.round(w_l2, 5)}   AdamW final w={np.round(w_adamw, 5)}")
print(f"Different endpoints: {not np.allclose(w_l2, w_adamw, atol=1e-4)} -- a real algorithmic change.\n")

# 2) The same update rule, generalized to a list of tensors, trains a real torch.nn network
class AdamForTorch:  # one (m,v) pair per parameter tensor
    def __init__(self, params, lr, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = list(params)
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0
    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * p.grad
                self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * p.grad**2
                m_hat, v_hat = self.m[i] / (1 - self.b1**self.t), self.v[i] / (1 - self.b2**self.t)
                p -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

rng_net = np.random.default_rng(0)
X_net = torch.tensor(rng_net.uniform(-2, 2, size=(100, 2)), dtype=torch.float32)
y_net = torch.tensor(np.sin(2 * X_net[:, 0].numpy()) + np.cos(X_net[:, 1].numpy()), dtype=torch.float32).unsqueeze(1)
loss_fn = torch.nn.MSELoss()

def make_and_train(optimizer_ctor):
    torch.manual_seed(1)
    model = torch.nn.Sequential(torch.nn.Linear(2, 8), torch.nn.Tanh(), torch.nn.Linear(8, 1))
    opt = optimizer_ctor(model.parameters())
    for _ in range(300):
        opt.zero_grad()
        loss_fn(model(X_net), y_net).backward()
        opt.step()
    return model

model_scratch = make_and_train(lambda p: AdamForTorch(p, lr=0.01))
model_ref = make_and_train(lambda p: torch.optim.Adam(p, lr=0.01))
params_match = all(torch.allclose(a, b, atol=1e-5) for a, b in zip(model_scratch.parameters(), model_ref.parameters()))
print(f"from-scratch Adam on a real torch.nn net: loss={loss_fn(model_scratch(X_net), y_net).item():.6f}")
print(f"torch.optim.Adam,   same net, same init: loss={loss_fn(model_ref(X_net), y_net).item():.6f}")
print(f"every parameter matches: {params_match}")

## 🏋️ Exercises

### Warm-up
1. For a single parameter with constant gradient $g=1.0$, $\beta_1=0.9$, $\beta_2=0.999$,
   $\eta=0.1$, compute $m_1$, $v_1$, $\hat{m}_1$, $\hat{v}_1$, and the resulting step by hand at
   $t=1$. Confirm the step equals $\eta$ exactly. Verify with the `Adam` class.

### Practice
2. In the bias-correction ablation cell, extend the loop to $t=1,\dots,50$ and plot both the
   corrected and uncorrected effective step size against $t$. You should find the uncorrected
   step stays well ABOVE $\eta$ for all 50 steps (it takes roughly $t=3900$ with the default
   $\beta_1=0.9,\beta_2=0.999$ to come within 1%). Now hold $\beta_1=0.9$ fixed and try
   $\beta_2=0.99$ instead: does the threshold get much smaller or much larger? Explain why the
   threshold is governed mainly by $\beta_2$ here, not $\beta_1$ — $v_t$ sits under a square
   root in the *denominator*, and $\beta_2$ being much closer to 1 than $\beta_1$ makes $v_t$'s
   bias factor $1-\beta_2^t$ converge far slower than $m_t$'s $1-\beta_1^t$, so the denominator
   stays under-corrected long after the numerator has essentially self-corrected.

### Challenge
3. Modify the `torch.optim` cross-check to use `betas=(0.5, 0.9)` for both our `Adam` class and
   `torch.optim.Adam`, confirm they still match, and then re-run the ill-conditioned-quadratic
   race from the Visual Intuition cell with these more aggressive betas. Explain why $\beta_2$
   closer to $0$ makes Adam's per-coordinate scaling noisier, connecting back to RMSProp's own
   $\beta$ in the Theory section.

---

## 📚 Further Reading
- Kingma & Ba, ["Adam: A Method for Stochastic Optimization"](https://arxiv.org/abs/1412.6980)
- Loshchilov & Hutter, ["Decoupled Weight Decay Regularization"](https://arxiv.org/abs/1711.05101) (AdamW)
- Duchi, Hazan & Singer, ["Adaptive Subgradient Methods for Online Learning and Stochastic Optimization"](https://jmlr.org/papers/v12/duchi11a.html) (AdaGrad)
- Sutskever et al., ["On the Importance of Initialization and Momentum in Deep Learning"](https://proceedings.mlr.press/v28/sutskever13.html)

---

*Next notebook: [Constrained Optimization](05_constrained_optimization.ipynb)*